# S2 v1 — IS diagnosis (sealed OOS off-limits)

Forensics on the frozen Universe E STAR using **research IS only**.

Check excess kurtosis, pair concentration, cost drag, and zero-volume skips **before** opening `02_oos_tearsheet.ipynb`. Disposition: kill / paper / new H. Do not re-tune on sealed OOS.


## 0. Imports & Config


In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.research import (
    DEFAULT_STAR_STACK,
    config_from_stack,
    is_end_for_stack,
    load_star_panels,
    load_universe_panels,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.star_stack_io import load_star_stack, require_star
from performance.sharpe_inference import return_moments
from strategies.s2_coint.metrics import metrics_from_returns

stack = load_star_stack(DEFAULT_STAR_STACK)
print("UNIVERSE", require_star(stack, "UNIVERSE_STAR"))
print("SIZE", stack.get("SIZE_STAR"), "OVERLAP", stack.get("OVERLAP_STAR"))
print("Disposition: kill / paper / new H — never re-tune on sealed OOS")


## 1. Research IS book


In [ ]:
universe = str(require_star(stack, "UNIVERSE_STAR"))
bar = str(stack.get("BAR_STAR") or "1d")
pairs = stack.get("PAIRS_STAR")
if not pairs:
    csv_path = os.path.join(ROOT, "01_data", "data_files", "s2_coint", f"s2_pairs_{universe}_{bar}.csv")
    pairs = pd.read_csv(csv_path)["pair_id"].astype(str).tolist()
try:
    train, full, manifest = load_star_panels(universe=universe, bar=bar, pair_ids=pairs, root=ROOT)
    panel = train
    print("loaded star train panel", manifest.get("stack_hash"))
except Exception as exc:
    warnings.warn(f"star panels unavailable ({exc}); loading universe panels")
    train, full = load_universe_panels(universe, bar, pairs, root=ROOT)
    panel = train if train is not None else full
is_end = is_end_for_stack(stack, panel)
panel = panel.loc[pd.to_datetime(panel["date"]) <= pd.Timestamp(is_end)].copy()
assert (pd.to_datetime(panel["date"]) > pd.Timestamp(is_end)).sum() == 0
cfg = config_from_stack(stack)
res = run_s2_backtest(panel, cfg)
print("IS end", is_end, "days", int(res.returns.shape[0]))


## 2. Tails, drawdown, gross vs net


In [ ]:
net = metrics_from_returns(res.returns)
mom = return_moments(res.returns)
excess = float(mom["kurtosis"] - 3.0) if np.isfinite(mom["kurtosis"]) else float("nan")
gross_s = res.book.returns_gross
gross = metrics_from_returns(gross_s) if gross_s is not None and not gross_s.empty else {}
row = {
    "ann_sharpe_net": net["ann_sharpe"],
    "ann_sharpe_gross": gross.get("ann_sharpe", float("nan")),
    "max_drawdown": net["max_drawdown"],
    "skew": mom["skew"],
    "excess_kurtosis": excess,
    "psr_input_sr_period": mom["sr_period"],
}
display(pd.Series(row, name="IS"))
if np.isfinite(excess) and excess > 8:
    warnings.warn(f"IS excess kurtosis {excess:.2f} — inspect a single pair before OOS")
